In [1]:
import gc
import json
import sqlite3
import uuid
from pathlib import Path

from qdrant_client import QdrantClient, models
from tuutrag.data import D

In [2]:
DB_PATH: Path = D.processed / "master.db"

QDRANT_HOST: str      = "localhost"
QDRANT_PORT: int      = 6333
EMBEDDING_DIM: int    = 3072
DISTANCE: models.Distance = models.Distance.COSINE

BRANCH_COLLECTION: str = "branches"
LEAF_COLLECTION: str   = "leafs"

UPSERT_BATCH: int = 256
SQL_FETCH: int = UPSERT_BATCH

In [3]:
qclient = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT, timeout=120, https=False)
print(f"Connected to Qdrant at http://{QDRANT_HOST}:{QDRANT_PORT}/dashboard")

Connected to Qdrant at http://localhost:6333/dashboard


In [4]:
for name in (BRANCH_COLLECTION, LEAF_COLLECTION):
    if qclient.collection_exists(name):
        qclient.delete_collection(name)

    qclient.create_collection(
        collection_name=name,
        vectors_config=models.VectorParams(
            size=EMBEDDING_DIM,
            distance=DISTANCE,
        ),
    )
    print(f"Collection '{name}' created ({EMBEDDING_DIM}-dim, {DISTANCE.name})")

Collection 'branches' created (3072-dim, COSINE)
Collection 'leafs' created (3072-dim, COSINE)


In [5]:
qclient.create_payload_index(BRANCH_COLLECTION, "artifact_uuid",
                             field_schema=models.PayloadSchemaType.KEYWORD)
qclient.create_payload_index(BRANCH_COLLECTION, "type",
                             field_schema=models.PayloadSchemaType.KEYWORD)

qclient.create_payload_index(LEAF_COLLECTION, "branch_uuid",
                             field_schema=models.PayloadSchemaType.KEYWORD)
qclient.create_payload_index(LEAF_COLLECTION, "artifact_uuid",
                             field_schema=models.PayloadSchemaType.KEYWORD)

print("Payload indexes created.")

Payload indexes created.


In [6]:
def qdrant_point_id(text_uuid: str) -> str:
    """Deterministic point ID from the text UUID."""
    return str(uuid.uuid5(uuid.NAMESPACE_DNS, text_uuid))

def iter_rows(cursor: sqlite3.Cursor, size: int = SQL_FETCH):
    """Yield rows in pages from a cursor."""
    while True:
        rows = cursor.fetchmany(size)
        if not rows:
            break
        yield rows

In [7]:
conn = sqlite3.connect(str(DB_PATH))
conn.row_factory = sqlite3.Row

cur = conn.cursor()
cur.execute("""
    SELECT
        be.branch_uuid,
        be.embedding,
        b.artifact_uuid,
        b.chunk,
        b.path,
        a.type
    FROM branch_embeddings  be
    JOIN branches           b  ON b.uuid = be.branch_uuid
    JOIN artifacts          a  ON a.uuid = b.artifact_uuid
""")

branch_count = 0

for page in iter_rows(cur):
    points = []
    for row in page:
        vec = json.loads(row["embedding"])
        points.append(
            models.PointStruct(
                id=qdrant_point_id(row["branch_uuid"]),
                vector=vec,
                payload={
                    "uuid":          row["branch_uuid"],
                    "artifact_uuid": row["artifact_uuid"],
                    "type":          row["type"],
                    "chunk":         row["chunk"],
                    "path":          row["path"],
                },
            )
        )
    qclient.upsert(collection_name=BRANCH_COLLECTION, points=points, wait=True)
    branch_count += len(points)
    del points, page
    gc.collect()

    if branch_count % (UPSERT_BATCH * 10) == 0:
        print(f"  branches upserted: {branch_count:,}")

cur.close()
print(f"Branch upsert complete — {branch_count:,} points")

  branches upserted: 2,560
  branches upserted: 5,120
  branches upserted: 7,680
  branches upserted: 10,240
  branches upserted: 12,800
  branches upserted: 15,360
  branches upserted: 17,920
  branches upserted: 20,480
  branches upserted: 23,040
  branches upserted: 25,600
  branches upserted: 28,160
  branches upserted: 30,720
  branches upserted: 33,280
  branches upserted: 35,840
Branch upsert complete — 37,239 points


In [8]:
cur = conn.cursor()
cur.execute("""
    SELECT
        le.leaf_uuid,
        le.embedding,
        l.branch_uuid,
        l.text,
        b.artifact_uuid,
        b.path,
        a.type
    FROM leaf_embeddings  le
    JOIN leafs            l  ON l.uuid        = le.leaf_uuid
    JOIN branches         b  ON b.uuid        = l.branch_uuid
    JOIN artifacts        a  ON a.uuid        = b.artifact_uuid
""")

leaf_count = 0

for page in iter_rows(cur):
    points = []
    for row in page:
        vec = json.loads(row["embedding"])
        points.append(
            models.PointStruct(
                id=qdrant_point_id(row["leaf_uuid"]),
                vector=vec,
                payload={
                    "uuid":          row["leaf_uuid"],
                    "branch_uuid":   row["branch_uuid"],
                    "artifact_uuid": row["artifact_uuid"],
                    "type":          row["type"],
                    "text":          row["text"],
                    "path":          row["path"],
                },
            )
        )
    qclient.upsert(collection_name=LEAF_COLLECTION, points=points, wait=True)
    leaf_count += len(points)
    del points, page
    gc.collect()

    if leaf_count % (UPSERT_BATCH * 10) == 0:
        print(f"  leafs upserted: {leaf_count:,}")

cur.close()
conn.close()
print(f"Leaf upsert complete — {leaf_count:,} points")

  leafs upserted: 2,560
  leafs upserted: 5,120
  leafs upserted: 7,680
  leafs upserted: 10,240
  leafs upserted: 12,800
  leafs upserted: 15,360
  leafs upserted: 17,920
  leafs upserted: 20,480
  leafs upserted: 23,040
  leafs upserted: 25,600
  leafs upserted: 28,160
  leafs upserted: 30,720
  leafs upserted: 33,280
  leafs upserted: 35,840
  leafs upserted: 38,400
  leafs upserted: 40,960
  leafs upserted: 43,520
  leafs upserted: 46,080
  leafs upserted: 48,640
  leafs upserted: 51,200
  leafs upserted: 53,760
  leafs upserted: 56,320
  leafs upserted: 58,880
  leafs upserted: 61,440
  leafs upserted: 64,000
  leafs upserted: 66,560
  leafs upserted: 69,120
  leafs upserted: 71,680
  leafs upserted: 74,240
  leafs upserted: 76,800
  leafs upserted: 79,360
  leafs upserted: 81,920
  leafs upserted: 84,480
  leafs upserted: 87,040
  leafs upserted: 89,600
  leafs upserted: 92,160
  leafs upserted: 94,720
  leafs upserted: 97,280
  leafs upserted: 99,840
  leafs upserted: 102,400
  

In [10]:
for name in (BRANCH_COLLECTION, LEAF_COLLECTION):
    info = qclient.get_collection(name)
    exact = qclient.count(collection_name=name, exact=True).count
    print(f"\n{name}:")
    print(f"  points:       {info.points_count:,}")
    print(f"  exact count:  {exact:,}")
    print(f"  index status: {info.status}")


branches:
  points:       37,239
  exact count:  37,239
  index status: green

leafs:
  points:       322,120
  exact count:  322,120
  index status: green


In [11]:
def search_similar_branches(
    branch_uuid: str,
    k: int = 10,
    type_filter: str | None = None,
) -> list[models.ScoredPoint]:
    """
    Find top-k branches most similar to `branch_uuid`.

    Parameters
    ----------
    branch_uuid : e.g. "041f16b3-…d300.1"
    k           : number of results
    type_filter : optional — restrict results to a specific artifact type
                  (e.g. "supplemental", "requirement", "test")
    """
    point_id = qdrant_point_id(branch_uuid)

    query_filter = None
    if type_filter:
        query_filter = models.Filter(
            must=[
                models.FieldCondition(
                    key="type",
                    match=models.MatchValue(value=type_filter),
                )
            ]
        )

    return qclient.query_points(
        collection_name=BRANCH_COLLECTION,
        query=point_id,
        using=None,
        limit=k,
        query_filter=query_filter,
        with_payload=True,
    ).points


def search_similar_leafs(
    leaf_uuid: str,
    k: int = 10,
    artifact_uuid_filter: str | None = None,
    branch_uuid_filter: str | None = None,
) -> list[models.ScoredPoint]:
    """
    Find top-k leafs most similar to `leaf_uuid`.

    Parameters
    ----------
    leaf_uuid             : e.g. "041f16b3-…d300.1.1"
    k                     : number of results
    artifact_uuid_filter  : restrict to leafs under a specific artifact
    branch_uuid_filter    : restrict to leafs under a specific branch
    """
    point_id = qdrant_point_id(leaf_uuid)

    conditions = []
    if artifact_uuid_filter:
        conditions.append(
            models.FieldCondition(
                key="artifact_uuid",
                match=models.MatchValue(value=artifact_uuid_filter),
            )
        )
    if branch_uuid_filter:
        conditions.append(
            models.FieldCondition(
                key="branch_uuid",
                match=models.MatchValue(value=branch_uuid_filter),
            )
        )

    query_filter = models.Filter(must=conditions) if conditions else None

    return qclient.query_points(
        collection_name=LEAF_COLLECTION,
        query=point_id,
        using=None,
        limit=k,
        query_filter=query_filter,
        with_payload=True,
    ).points

In [14]:
results = search_similar_branches("041f16b3-f3b7-488f-a27c-074e63a9d300.1", k=5)
for r in results:
    print(f"  {r.score:.4f}  {r.payload['uuid']}  {r.payload['type']}    {r.payload['chunk']}")

  0.9999  58460460-bc83-40cc-ae02-e24d41e050ac.1  supplemental    [{'dataDictionary': {'LDD Version': '1.24.0.0', 'Description': 'This document is a dump of the contents of the PDS4 Data Dictionary', 'IM Version': '1.24.0.0', 'ExternTermMapDictionary': [{'infoModelVersionId': '1.24.0.0', 'datetime': '2025-04-28T22:35:24Z', 'lddVersion': None, 'termmaps': [], 'title': 'PDS4 Term Mappings', 'lddName': None}], 'Title': 'PDS4 Data Dictionary', 'dataTypeDictionary': [{'DataType': {'minimumValue': 'Unbounded', 'identifier': '0001_NASA_PDS_1.pds.ASCII_AnyURI', 'registrationAuthorityId': '0001_NASA_PDS_1', 'minimumCharacters': 'Unbounded', 'maximumCharacters': 'Unbounded', 'nameSpaceId': 'pds', 'title': 'ASCII_AnyURI', 'maximumValue': 'Unbounded'}}, {'DataType': {'minimumValue': 'Unbounded', 'identifier': '0001_NASA_PDS_1.pds.ASCII_BibCode', 'registrationAuthorityId': '0001_NASA_PDS_1', 'minimumCharacters': 'Unbounded', 'maximumCharacters': 'Unbounded', 'nameSpaceId': 'pds', 'title': 'ASCII_Bi

In [13]:
results = search_similar_leafs(
    "041f16b3-f3b7-488f-a27c-074e63a9d300.1.1",
    k=5,
    artifact_uuid_filter="041f16b3-f3b7-488f-a27c-074e63a9d300"
)
for r in results:
    print(f"  {r.score:.4f}  {r.payload['uuid']}  {r.payload['branch_uuid']}")

  0.9214  041f16b3-f3b7-488f-a27c-074e63a9d300.547.6  041f16b3-f3b7-488f-a27c-074e63a9d300.547
  0.9153  041f16b3-f3b7-488f-a27c-074e63a9d300.6.4  041f16b3-f3b7-488f-a27c-074e63a9d300.6
  0.9145  041f16b3-f3b7-488f-a27c-074e63a9d300.191.7  041f16b3-f3b7-488f-a27c-074e63a9d300.191
  0.9141  041f16b3-f3b7-488f-a27c-074e63a9d300.8.2  041f16b3-f3b7-488f-a27c-074e63a9d300.8
  0.9138  041f16b3-f3b7-488f-a27c-074e63a9d300.2.6  041f16b3-f3b7-488f-a27c-074e63a9d300.2
